In [ ]:
from langchain_ollama import ChatOllama
############################################################################################################################################################
# DEMOCRATIC WORKFLOW
############################################################################################################################################################
from sklearn.metrics import classification_report
from tools import ConfusionMatrix
from main import SummaryTMP
import pandas as pd
import json
import matplotlib.pyplot as plt

# State rows to keep
keep=['accession', 'thermal_range', 'thermal_source', 'thermal_reasoning', 'thermal_confidence', 'thermal_source', 'inference_type', 'thermal_paper', 'host','taxonomic_level','host_reasoning','host_confidence','host_source','host_paper','duration', 'nodes', 'timings']

#Getting accessions
ex_df=pd.read_csv('data/example/expanded_example_data.csv')
accessions=ex_df['Accession']


all_df = pd.DataFrame()

# Classifying each accession
preds=[]
for acc in accessions:
    print(f'Accession: {acc}')
    result = SummaryTMP(acc, "gemma4:e4b")


    preds.append(result.get('thermal_range', None))
    print(f'\n{acc}: {result.get("thermal_range", None)}')
    df = pd.DataFrame([result])

    # serialize complex fields
    df["metadata"] = df["metadata"].apply(json.dumps)
    df["timings"] = df["timings"].apply(json.dumps)
    df["nodes"] = df["nodes"].apply(json.dumps)

    all_df = pd.concat([all_df, df], ignore_index=True)



all_df["thermal_range"] = preds

all_df.to_csv('results/summary_result.csv', index=False)

# Create confusion matrix
actual=ex_df['Thermal Range']
pred=all_df['thermal_range']

#Clean out None
actual_clean = actual.fillna("unknown")
pred_clean = pred.fillna("unknown")

plot=ConfusionMatrix(actual_clean, pred_clean)
plot.plot(cmap='Blues')
plt.savefig('results/summary_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()


# Total duration
import pandas as pd
df=pd.read_csv('results/summary_result.csv')
total_duration=0
for i in df['duration']:
    total_duration+=i
print(f'Total Duration: {round((total_duration/60),2)} minutes')

# Get the classification report
classification_report=classification_report(actual_clean, pred_clean)

with open('results/summary_classification_report.txt', 'w') as f:
    f.write(classification_report)
    f.write(f'\n\n Total Duration: {round((total_duration/60),2)} minutes')


In [ ]:
from graphs import VisualiseGraph, SummaryGraph
VisualiseGraph(SummaryGraph)